# Plot panels

<div align="center">
<img src="https://github.com/SciQLop/SciQLop/raw/main/SciQLop/resources/icons/SciQLop.png"/>
</div>

This tutorial covers the Python API for creating and manipulating SciQLop plot panels from a notebook: adding products, configuring axes, building multi-subplot layouts, overlaying products, exporting images, and saving/loading panel templates. It introduces both the **imperative API** (step-by-step `create_plot_panel` + `.plot(...)`) and the **fluent API** (a chainable builder).

**What you'll learn**
- Create a plot panel and plot a product from Python.
- Change axis scales and explicit Y bounds.
- Build multi-subplot layouts and overlay multiple products on the same axes.
- Export a panel to PNG/PDF.
- Save a panel layout as a template and re-instantiate it later.

**Prerequisites** — [SciQLop GUI tour](1-SciQLopGUIDiscovery.ipynb), [Speasy first steps](../Speasy/2-SpeasyFirstSteps.ipynb).

**Next** — [IPython magics](7-SciQLopMagics.ipynb) to do all of this with one-liners, then [Virtual products](6-SciQLopVirtualProducts.ipynb).

## 1. Basic plotting


In [ ]:
from SciQLop.user_api import TimeRange
from SciQLop.user_api.plot import create_plot_panel

p = create_plot_panel()
p.time_range = TimeRange("2015-11-18T02:14:30", "2015-11-18T03:34:00")

p.plot("speasy/cda/MMS/MMS1/FGM/MMS1_FGM_SRVY_L2/mms1_fgm_b_gse_srvy_l2")


A product path is the string you'd see if you hovered over a leaf in the product tree. The convention is `provider/<subtree>/<dataset>/<variable>`.

> **Tip — drag from the tree.** Inside SciQLop's embedded JupyterLab you can drag a product from the product tree into the cell below between the `""` characters; the path is inserted automatically.


In [ ]:
density_plot, _ = p.plot("speasy/cda/MMS/MMS1/DIS/MMS1_FPI_FAST_L2_DIS_MOMS/mms1_dis_numberdensity_fast")


## 2. Axis scale and range

Change the Y-axis to logarithmic and set explicit bounds.

In [ ]:
from SciQLop.user_api.plot import ScaleType

density_plot.y_scale_type = ScaleType.Logarithmic
density_plot.set_y_range(0.1, 100)

## 3. The fluent API

`fluent.new_panel()` returns a chainable builder. Each call returns the builder, so you can describe the whole panel in a single expression. The two examples below build the same panel imperatively and fluently.

Pick whichever style you prefer — they share the same underlying objects.


In [ ]:
from SciQLop.user_api.plot import fluent

builder = (fluent.new_panel()
    .plot("speasy/cda/MMS/MMS1/FGM/MMS1_FGM_SRVY_L2/mms1_fgm_b_gse_srvy_l2")
    .subplot()
        .plot("speasy/cda/MMS/MMS1/DIS/MMS1_FPI_FAST_L2_DIS_MOMS/mms1_dis_numberdensity_fast")
        .log_y()
        .y_range(0.1, 100)
    .time_range("2015-11-18T02:14:30", "2015-11-18T03:34:00"))


### Multi-subplot panels

Use `.subplot()` to start a new set of axes. Calls between two `.subplot()`s land on the most-recent subplot.


In [ ]:
overview = (fluent.new_panel()
    # 1. Magnetic field
    .plot("speasy/cda/MMS/MMS1/FGM/MMS1_FGM_SRVY_L2/mms1_fgm_b_gse_srvy_l2")
    .y_range(-75, 75)

    # 2. Ion bulk velocity
    .subplot()
    .plot("speasy/cda/MMS/MMS1/DIS/MMS1_FPI_FAST_L2_DIS_MOMS/mms1_dis_bulkv_gse_fast")
    .y_range(-400, 400)

    # 3. Number density (log scale)
    .subplot()
    .plot("speasy/cda/MMS/MMS1/DIS/MMS1_FPI_FAST_L2_DIS_MOMS/mms1_dis_numberdensity_fast")
    .log_y()
    .y_range(0.01, 100)

    .time_range("2015-11-18T02:14:30", "2015-11-18T03:34:00"))


### Overlaying products on the same plot

Two `.plot()` calls without an intervening `.subplot()` overlay the products on the same axes.


In [ ]:
overlay = (fluent.new_panel()
    # Compare survey-mode and burst-mode density on the same axes
    .plot("speasy/cda/MMS/MMS1/DIS/MMS1_FPI_FAST_L2_DIS_MOMS/mms1_dis_numberdensity_fast")
    .plot("speasy/cda/MMS/MMS1/DIS/MMS1_FPI_BRST_L2_DIS_MOMS/mms1_dis_numberdensity_brst")
    .log_y()
    .y_range(0.01, 100)
    .time_range("2015-11-18T02:14:30", "2015-11-18T03:34:00"))


### Falling back to imperative access

The fluent builder owns a `PanelBuilder`; if you ever need the underlying `PlotPanel` for an imperative operation, use `.panel`. And to wrap an already-open panel by its tab name, use `fluent.panel(name)`.


In [ ]:
overview_panel = overview.panel
print(f"Plots in panel: {len(overview_panel.plots)}")


## 4. Exporting plots

`PlotPanel.save(path)` writes the panel to disk. Supported formats are PNG, PDF, JPG and BMP — picked from the file extension.


In [ ]:
from pathlib import Path
import tempfile

out_dir = Path(tempfile.gettempdir())
png_path = out_dir / "mms_overview.png"
overview_panel.save(str(png_path))
print(f"Wrote {png_path} ({png_path.stat().st_size} bytes)")


## 5. Panel templates

A **template** captures a panel's layout (subplot count, axis scales, products) without its data. You can re-instantiate it later — useful for standard quick-look layouts.


In [ ]:
from SciQLop.user_api import templates

overview_panel.save_template("mms_overview")

for t in templates.list_templates():
    print(t.name)


In [ ]:
# Load the template back (returns a PanelTemplate object you can instantiate from the GUI)
template = templates.load("mms_overview")
print(f"Loaded template: {template.name}")

# Rename and delete are also available:
# templates.rename("mms_overview", "mms_overview_v2")
# templates.delete("mms_overview_v2")


## Fluent API reference

| Method | Effect |
|--------|--------|
| `fluent.new_panel()` | Create a new panel |
| `fluent.panel(name)` | Wrap an existing panel by tab name |
| `.plot(product, **kw)` | Add a graph (same args as `PlotPanel.plot()`) |
| `.subplot()` | Start a new subplot |
| `.y_range(lo, hi)` | Set Y bounds on the current subplot |
| `.log_y()` / `.linear_y()` | Set Y scale type |
| `.layer(func, **kw)` | Attach an annotation layer (see [tutorial 16](16-SciQLopAnnotationLayers.ipynb)) |
| `.time_range(start, stop)` | Set the panel's time range |
| `.panel` | Get the underlying `PlotPanel` for imperative ops |
